# Notebook 3: RLHF Analysis
Inspect reward model quality, preference data statistics, and RLHF training curves.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path

## 1. Preference data statistics

In [ ]:
prefs = []
pref_file = Path('../data/preferences/sample_preferences.jsonl')
with open(pref_file) as f:
    for line in f:
        prefs.append(json.loads(line))

df = pd.DataFrame(prefs)
print(f'Total preference pairs: {len(df)}')

df['chosen_len'] = df['chosen'].str.len()
df['rejected_len'] = df['rejected'].str.len()
df['len_ratio'] = df['chosen_len'] / df['rejected_len'].clip(lower=1)

print('\nChosen response length stats:')
print(df['chosen_len'].describe())
print('\nChosenlen / Rejectedlen ratio (>1 = chosen is longer):')
print(df['len_ratio'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[['chosen_len', 'rejected_len']].plot(kind='hist', bins=30, alpha=0.6, ax=axes[0])
axes[0].set_title('Response length: Chosen vs Rejected')
df['len_ratio'].plot(kind='hist', bins=30, ax=axes[1], color='purple')
axes[1].set_title('Length ratio (chosen/rejected)')
plt.tight_layout()
plt.show()

## 2. Reward model scoring — sanity check

In [ ]:
# Verify reward model gives higher scores to 'chosen' than 'rejected'
# (Only run this after training the reward model)

REWARD_MODEL_PATH = '../models/reward'

if Path(REWARD_MODEL_PATH).exists():
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    from src.rlhf.reward_model import compute_rewards

    rm = AutoModelForSequenceClassification.from_pretrained(
        REWARD_MODEL_PATH, torch_dtype=torch.bfloat16, num_labels=1
    )
    rm_tok = AutoTokenizer.from_pretrained(REWARD_MODEL_PATH)
    rm.eval()

    results = []
    for row in prefs[:10]:
        r_chosen = rm(rm_tok(row['prompt'] + row['chosen'],
            return_tensors='pt', truncation=True, max_length=512)
        ).logits.item()
        r_rejected = rm(rm_tok(row['prompt'] + row['rejected'],
            return_tensors='pt', truncation=True, max_length=512)
        ).logits.item()
        results.append({'chosen_reward': r_chosen, 'rejected_reward': r_rejected, 'correct': r_chosen > r_rejected})

    res_df = pd.DataFrame(results)
    print(f"Reward model accuracy (chosen > rejected): {res_df['correct'].mean():.1%}")
    print(res_df[['chosen_reward', 'rejected_reward', 'correct']].round(3))
else:
    print(f'Reward model not found at {REWARD_MODEL_PATH}. Train it first.')

## 3. PPO training curves (from W&B / TensorBoard logs)

In [ ]:
# If using W&B, pull the run data:
# import wandb
# api = wandb.Api()
# run = api.run('<entity>/<project>/<run_id>')
# history = run.history()
# history[['_step', 'objective/reward', 'objective/kl']].plot(x='_step', figsize=(12,4))
print('Replace this cell with W&B run ID to plot training curves.')

## 4. Side-by-side: SFT vs RLHF outputs

In [ ]:
SFT_PATH = '../models/finetuned'
RLHF_PATH = '../models/rlhf_final'

TEST_PROMPTS = [
    'What is the doctrine of promissory estoppel?',
    'Explain the key provisions of a non-disclosure agreement.',
    'What does "material breach" mean in contract law?',
]

if Path(SFT_PATH).exists() and Path(RLHF_PATH).exists():
    from src.inference import LLMInference

    sft_engine = LLMInference(model_path=SFT_PATH, domain='legal')
    rlhf_engine = LLMInference(model_path=RLHF_PATH, domain='legal')

    for prompt in TEST_PROMPTS:
        print('=' * 70)
        print(f'PROMPT: {prompt}')
        print('-' * 35 + ' SFT ' + '-' * 35)
        sft_resp = sft_engine.generate(prompt, max_new_tokens=200)
        print(sft_resp)
        print('-' * 34 + ' RLHF ' + '-' * 34)
        rlhf_resp = rlhf_engine.generate(prompt, max_new_tokens=200)
        print(rlhf_resp)
else:
    print('Train both models first, then re-run this cell.')